# Smoke Test

Verifies: clean imports, 10-second generation, 10-step LoRA benchmark.

**Run this before any other notebook.**

In [ ]:
# Cell 1: Install pinned dependencies
import subprocess
subprocess.run('pip install -q transformers==4.57.6 huggingface_hub==0.36.0 '
               'accelerate==1.12.0 "peft==0.18.*"', shell=True, check=True)
subprocess.run('pip install -q --no-deps '
               'git+https://github.com/jthickstun/anticipation.git', shell=True, check=True)
subprocess.run('pip install -q mido==1.2.10 pretty_midi scipy scikit-learn', shell=True, check=True)
print('Installs done.')

In [ ]:
# Cell 2: Verify imports and record library versions
import torch, transformers, peft, anticipation
print(f'torch       {torch.__version__}')
print(f'transformers {transformers.__version__}')
print(f'peft        {peft.__version__}')
print(f'cuda        {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 3: Inspect anticipation config — record CONTROL_OFFSET and vocab size
import anticipation.config as acfg
print('CONTROL_OFFSET =', acfg.CONTROL_OFFSET)  # UNVERIFIED until run
print('anticipation config attrs:', [a for a in dir(acfg) if not a.startswith('_')])

In [ ]:
# Cell 4: Load model in fp16
# Try AutoModelForCausalLM first; fall back to GPT2LMHeadModel if architectures field fails.
import torch
from transformers import AutoTokenizer

MODEL_NAME = 'stanford-crfm/music-medium-800k'

try:
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    print('Loaded with AutoModelForCausalLM')
except Exception as e:
    print(f'AutoModelForCausalLM failed ({e}), trying GPT2LMHeadModel')
    from transformers import GPT2LMHeadModel
    model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    print('Loaded with GPT2LMHeadModel')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print(f'vocab_size = {model.config.vocab_size}')  # record this
print(f'n_layer    = {model.config.n_layer}')
print(f'n_embd     = {model.config.n_embd}')
print(f'Parameters = {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

In [ ]:
# Cell 5: Generate a short clip (10 seconds) using anticipation
# Uses the library's own generate() with a simple C-major melody as controls.
import time
from anticipation import ops
from anticipation.sample import generate
from anticipation.convert import events_to_midi
import anticipation.config as acfg

# Build a minimal 4-note melody: C4 E4 G4 C5 at t=0,1,2,3 s, duration 0.8 s each
# Token format: (arrival_time_bin, duration_bin, pitch)
# UNVERIFIED: exact bin sizes — using seconds=True if available
# We pass melody as controls by adding CONTROL_OFFSET to each token
CO = acfg.CONTROL_OFFSET

# Build control tokens for melody (C4=60, E4=64, G4=67, C5=72)
# Using ops to build events then shift to control space
try:
    melody_events = ops.encode([
        (0.0, 0.8, 60), (1.0, 0.8, 64), (2.0, 0.8, 67), (3.0, 0.8, 72)
    ])
    controls = [t + CO for t in melody_events]
    print(f'Control tokens (first 12): {controls[:12]}')
except Exception as e:
    print(f'ops.encode not available: {e}')
    controls = []

torch.manual_seed(42)
t0 = time.time()
with torch.no_grad():
    output = generate(
        model,
        start_time=0,
        end_time=10,       # 10 seconds
        inputs=[],
        controls=controls,
        top_p=0.95,
    )
elapsed = time.time() - t0
print(f'Generation took {elapsed:.1f}s, output tokens: {len(output)}')

# Save MIDI
mid = events_to_midi(output)
mid.save('/content/smoke_test_output.mid')
print('Saved /content/smoke_test_output.mid')

In [ ]:
# Cell 6: 10-step LoRA benchmark
# Reports seconds/step and peak GPU memory.
import torch, time
from peft import LoraConfig, get_peft_model
from torch.cuda.amp import GradScaler, autocast

torch.cuda.reset_peak_memory_stats()

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['c_attn', 'c_proj', 'c_fc'],
    fan_in_fan_out=True,
    bias='none',
)

# Reload base model fresh for benchmark
try:
    from transformers import AutoModelForCausalLM
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)
except Exception:
    from transformers import GPT2LMHeadModel
    base = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)

base.enable_input_require_grads()
base.gradient_checkpointing_enable()
lora_model = get_peft_model(base, lora_cfg)
lora_model.print_trainable_parameters()

optimizer = torch.optim.AdamW(
    [p for p in lora_model.parameters() if p.requires_grad], lr=2e-4
)
scaler = GradScaler()

SEQ_LEN = 512
BATCH = 4
vocab = base.config.vocab_size

times = []
for step in range(10):
    ids = torch.randint(0, vocab, (BATCH, SEQ_LEN), device=device)
    t0 = time.time()
    with autocast():
        out = lora_model(input_ids=ids, labels=ids)
        loss = out.loss
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()
    times.append(time.time() - t0)

peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f'Mean s/step: {sum(times)/len(times):.2f}')
print(f'Peak GPU memory: {peak_gb:.2f} GB')
print('Smoke test PASSED.')